# Reflectance training samples — manual ROIs, patch averaging

You draw the white reference strip and each plate by hand, then every plate is cut into many small
patches. Each patch is averaged into one spectrum, so one plate becomes dozens of training rows instead
of a single mean.

**Output:** `training_samples.csv` — one row per patch, one column per wavelength (rounded), with the
composition labels attached.

## Why patches rather than individual pixels

At roughly 0.5–0.7 mm per pixel and coffee/chicory particles of 0.3–1.0 mm, a **single pixel is mostly one
particle** — it is coffee or chicory, not a 30 % mixture. Training on individual pixels asks the model to
predict a bulk property that the pixel does not physically have, and the resulting error is dominated by
particle lottery rather than chemistry.

A patch that spans several millimetres contains enough particles that its composition approximates the
bulk. That is what makes the sample meaningful. Section 8 reports your patch size in millimetres so you
can check it.

## The thing that will bite you later

Patches cut from one plate are **pseudo-replicates**. Same powder, same packing, same instant. They are
not independent measurements, however many of them you generate.

This matters when you split the data:

- **Random split → meaningless.** Near-identical sibling patches end up in both train and test. You will
  see R² above 0.99 and it will tell you nothing.
- **Split by group → honest.** The CSV carries a `group` column (scan + plate). Use `GroupKFold` or
  `LeaveOneGroupOut` on it, or better, hold out whole *concentration levels* so the model has to
  interpolate rather than recognise a dish it has already seen.

More patches gives the model more to fit, but your effective sample size is still the number of plates
you actually prepared, not the number of rows in the file.

## 1. Configuration

`pip install opencv-python` if you have not already — **not** `opencv-python-headless`, that build has no
GUI and the drawing window will fail to open.

In [ ]:
import os, re, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
import cv2

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "figure.facecolor": "white"})

# ------------------------------------ EDIT ------------------------------------
DATA_DIR  = Path(os.environ.get("COFFEE_RAW", "/run/media/riyanvarghese/RIYAN HDD"))
PANEL_TXT = Path("./SRT-MS-050__1_.txt")
OUT       = Path("./samples_out")

DAY1_PLATES = ["AB100", "CH100", "RB100", "A", "B", "C"]
DAY2_PLATES = ["RB[50:50]", "RB[60:40]", "RB[70:30]", "RB[80:20]", "RB[90:10]"]
DAY_RULES   = [("2026-09-01", 1, DAY1_PLATES), ("2026-09-04", 2, DAY2_PLATES)]

ROUND_WL   = 0      # decimals on the wavelength column names: 0 -> 952, 958, 963 ...

PATCH      = 12     # patch side in pixels
STRIDE     = 12     # == PATCH means non-overlapping. Smaller overlaps them (see section 8).
MIN_VALID  = 0.92   # a patch is kept only if this fraction of its pixels are inside the mask
MAX_PER_PLATE = 150 # cap; patches beyond this are randomly subsampled. None = keep all

ERODE_PX   = 4      # trimmed inward from the drawn ellipse
MM_PER_PX  = 0.73   # header nominal, used only to report patch size in mm

SEED = 0
# -------------------------------------------------------------------------------

OUT.mkdir(exist_ok=True, parents=True)
ROI_FILE = OUT / "rois.json"
RNG = np.random.default_rng(SEED)
print("OpenCV", cv2.__version__, "| data dir exists:", DATA_DIR.exists())

In [ ]:
ENVI_DTYPES = {1:"u1", 2:"i2", 3:"i4", 4:"f4", 5:"f8", 12:"u2", 13:"u4"}

def parse_envi_header(p):
    txt = Path(p).read_text(errors="ignore").replace("\r", "")
    body = txt.split("\n", 1)[1] if "\n" in txt else ""
    out, pos = {}, 0
    pat = re.compile(r"^\s*([\w\s]+?)\s*=\s*(\{.*?\}|[^\n]*)", re.S | re.M)
    while pos < len(body):
        m = pat.search(body, pos)
        if not m: break
        k, v = m.group(1).strip().lower(), m.group(2).strip()
        if v.startswith("{") and not v.endswith("}"):
            e = body.find("}", m.end(2)); v = body[m.start(2):e+1]; pos = e + 1
        else: pos = m.end(2)
        out[k] = v
    for k in ["samples","lines","bands","header offset","data type","byte order"]:
        if k in out: out[k] = int(float(out[k]))
    if "data ignore value" in out: out["data ignore value"] = float(out["data ignore value"])
    if "wavelength" in out:
        out["wavelength"] = np.array([float(x) for x in
            out["wavelength"].strip("{}").replace("\n"," ").split(",") if x.strip()])
    out["interleave"] = out.get("interleave","bsq").strip().lower()
    return out


class Cube:
    def __init__(self, hdr):
        self.hdr_path = Path(hdr); self.h = parse_envi_header(hdr)
        stem = self.hdr_path.with_suffix("")
        self.path = next((c for e in (".img",".raw",".dat",".bin","")
                          if (c := stem.with_suffix(e)).exists() and c != self.hdr_path), None)
        if self.path is None: raise FileNotFoundError(f"no data file beside {self.hdr_path.name}")
        L,S,B = self.h["lines"], self.h["samples"], self.h["bands"]
        dtc = ENVI_DTYPES[self.h["data type"]]
        nb = L*S*B*np.dtype(dtc).itemsize; fs = self.path.stat().st_size
        dec = self.h.get("header offset", 0)
        # the declared offset is not a whole number of BIL lines, so verify it against file size
        if   fs == dec + nb: off, why = dec, "declared offset matches"
        elif fs == nb:       off, why = 0, "no offset"
        elif fs > nb and fs - nb < 10_000_000: off, why = fs-nb, f"inferred {fs-nb}"
        else: raise ValueError(f"{self.path.name}: size {fs} vs {nb}+{dec}; file may be truncated")
        self.offset, self.offset_reason = off, why
        dt = np.dtype(dtc).newbyteorder("<" if self.h.get("byte order",0)==0 else ">")
        shp = {"bil":(L,B,S),"bsq":(B,L,S),"bip":(L,S,B)}[self.h["interleave"]]
        mm = np.memmap(self.path, dtype=dt, mode="r", offset=off, shape=shp)
        self.arr = {"bil":mm.transpose(0,2,1),"bsq":mm.transpose(1,2,0),
                    "bip":mm}[self.h["interleave"]]
        self.wl = self.h["wavelength"]; self.ignore = self.h.get("data ignore value", None)

    @property
    def shape(self): return self.arr.shape

    def region(self, r0, r1, c0=0, c1=None):
        c1 = self.shape[1] if c1 is None else c1
        b = np.array(self.arr[r0:r1, c0:c1, :], dtype=np.float32)   # np.array => writable copy
        if self.ignore is not None: b[b == self.ignore] = np.nan
        return b

    def brightness(self, step=1, lo=1500, hi=1700):
        idx = [i for i,w in enumerate(self.wl) if lo <= w <= hi][:8]
        s = np.array(self.arr[::step, ::step, :][:, :, idx], dtype=np.float32)
        if self.ignore is not None: s[s == self.ignore] = np.nan
        return np.nanmean(s, axis=2)


def load_white_cert(txt_path, wl):
    rows = []
    for line in Path(txt_path).read_text(errors="ignore").replace("\r","").split("\n"):
        p = line.strip().split("\t")
        if len(p) >= 2:
            try: rows.append([float(x) for x in p[:5]])
            except ValueError: continue
    a = np.array(rows)
    return np.interp(wl, a[:,0], a[:,1]), (wl < a[0,0]) | (wl > a[-1,0])


def capture_of(stem):
    # stir1 / stir2 / stir3 -> 1 / 2 / 3
    m = re.search(r"stir\s*(\d+)", stem, re.I)
    return int(m.group(1)) if m else None

CUBES, INFO = {}, {}
for h in sorted(DATA_DIR.rglob("*.hdr")):
    try:
        c = Cube(h); CUBES[h.stem] = c
        day, plates = next(((d,pl) for s,d,pl in DAY_RULES if s in h.stem), (None, []))
        INFO[h.stem] = dict(day=day, capture=capture_of(h.stem), plates=plates)
        print(f"  {h.stem}\n     {c.shape[0]}x{c.shape[1]}x{c.shape[2]} | {c.offset_reason} "
              f"| day {day}, stir {INFO[h.stem]['capture']}, {len(plates)} plates")
    except Exception as e:
        print(f"  !! {h.name}: {e}")

if any(v["capture"] is None for v in INFO.values()):
    print("\n!! Could not read a stir number from some filenames. Set INFO[key]['capture'] by hand,")
    print("   otherwise leave-one-stir-out cross-validation will not work.")
missing_day = [k for k, v in INFO.items() if v["day"] is None]
if missing_day:
    print("\n!! No day rule matched:", missing_day, "-> add a substring to DAY_RULES")

if not CUBES: raise SystemExit("no cubes opened -- check DATA_DIR")
k0 = list(CUBES)[0]
WL = CUBES[k0].wl.copy()
R_WHITE, OUTSIDE = load_white_cert(PANEL_TXT, WL)
WL_R = np.round(WL, ROUND_WL)
if ROUND_WL == 0: WL_R = WL_R.astype(int)
assert len(set(WL_R.tolist())) == WL.size, "rounding merged two bands -- raise ROUND_WL"
print(f"\n{WL.size} bands {WL[0]:.1f}-{WL[-1]:.1f} nm | labels {WL_R[0]}..{WL_R[-1]}")
print(f"white certificate: {R_WHITE[0]:.4f} -> {R_WHITE[-1]:.4f}  "
      f"({OUTSIDE.sum()} bands past the certificate's 2500 nm limit)")

## 2. Composition labels

Every sample carries its composition, so the CSV is ready for regression without further bookkeeping.
`chicory_pct` is the natural regression target. A, B and C get `NaN` — they are what you want to predict.

`AB100` is arabica rather than robusta, so it is flagged by `species` and kept out of the robusta series.

In [ ]:
def parse_label(lab):
    if lab == "CH100":  return dict(species="chicory",  robusta_pct=0.0,  chicory_pct=100.0, unknown=False)
    if lab == "RB100":  return dict(species="robusta",  robusta_pct=100.0, chicory_pct=0.0,  unknown=False)
    if lab == "AB100":  return dict(species="arabica",  robusta_pct=0.0,  chicory_pct=0.0,  unknown=False)
    m = re.match(r"RB\[(\d+):(\d+)\]", lab)
    if m:
        rb, ch = float(m.group(1)), float(m.group(2))
        return dict(species="robusta_chicory", robusta_pct=rb, chicory_pct=ch, unknown=False)
    return dict(species="unknown", robusta_pct=np.nan, chicory_pct=np.nan, unknown=True)

print(pd.DataFrame([dict(label=l, **parse_label(l))
                    for l in DAY1_PLATES + DAY2_PLATES]).to_string(index=False))

## 3. Drawing the white strip and the plates

The window opens with the **whole scan visible**. A 384 × 2000 cube at true proportions would be a sliver
about 190 px wide, so the two axes are scaled independently to fill your screen — the scan looks stretched
sideways and the round plates look like wide ovals. Everything is stored in original pixel coordinates,
so the stretch has no effect on the result. Press `2` for an undistorted 1:1 view to fine-tune.

**Draw the oval from its centre.** Press at the middle of a plate and drag outward; width and height
follow the drag. There is no bounding box anywhere in the interaction.

| Action | Effect |
|---|---|
| left-drag | draw from the centre outward |
| right-drag | move the current shape |
| `Shift`+drag | force a circle on screen |
| `n` / `p` | next / previous label |
| `r` / `e` / `g` | rectangle / ellipse / freehand mode |
| `1` / `2` | whole-scan view / 1:1 view |
| `j` / `k` | page down / up in 1:1 view |
| `w a s d` | nudge 1 px |
| `W A S D` | grow / shrink |
| `[` `]` | rotate 2° |
| `x` | delete current shape |
| ENTER | save and close |
| ESC / `q` | close without saving |

Two things worth being fussy about:

- **The WHITE rectangle** should sit well inside the bright strip and span as much of the swath width as
  you can. Its width is what lets the illumination profile be measured per column; its distance from the
  edges is what keeps the darker second strip out.
- **Plate ovals go off-centre, below the shadow crescent.** That black arc on each leading edge is the
  main contaminant — it darkens a spectrum the same way a composition change would. Losing a third of a
  plate costs you little; you still get plenty of patches.

In [ ]:
def screen_size(default=(1600, 900)):
    try:
        import tkinter as tk
        r = tk.Tk(); r.withdraw()
        s = (r.winfo_screenwidth(), r.winfo_screenheight()); r.destroy()
        return s
    except Exception:
        return default

SCREEN_W, SCREEN_H = screen_size()
VIEW_W, VIEW_H = int(SCREEN_W*0.86), int(SCREEN_H*0.80)
print(f"screen {SCREEN_W}x{SCREEN_H} -> window {VIEW_W}x{VIEW_H} (override VIEW_W/VIEW_H if wrong)")

def _u8(img, lo=1.5, hi=99.0):
    a = np.nan_to_num(img.astype(np.float32))
    l, h = np.percentile(a, [lo, hi])
    return np.clip((a-l)/max(h-l, 1e-9)*255, 0, 255).astype(np.uint8)

def ellipse_on_screen(s, sx, sy):
    # An ellipse stays an ellipse under independent x/y scaling, but its axes AND angle both
    # change. Carry it through the conic matrix so a rotated oval still draws where it belongs.
    th = np.deg2rad(s.get("angle", 0.0))
    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
    A = R @ np.diag([1/max(s["ax"],1e-6)**2, 1/max(s["ay"],1e-6)**2]) @ R.T
    Si = np.diag([1/max(sx,1e-9), 1/max(sy,1e-9)])
    w, V = np.linalg.eigh(Si @ A @ Si)
    w = np.maximum(w, 1e-12)
    return (float(1/np.sqrt(w[0])), float(1/np.sqrt(w[1]))), \
           float(np.degrees(np.arctan2(V[1,0], V[0,0])))

In [ ]:
def draw_rois(cube, key, plate_labels, existing=None):
    base = _u8(cube.brightness(step=1)); H, W = base.shape
    FIT = (VIEW_W/W, VIEW_H/H)
    ONE = (max(1.0, VIEW_W/W),) * 2
    labels = ["WHITE"] + list(plate_labels)
    default_kind = {l: ("rect" if l == "WHITE" else "ellipse") for l in labels}
    shapes = {k: dict(v) for k, v in (existing or {}).items()}
    st = dict(i=0, mode="fit", kind=None, top=0, draw=None, move=None, poly=None,
              save=False, abort=False)
    cache = {}

    def sxsy(): return FIT if st["mode"] == "fit" else ONE
    def canvas():
        sx, sy = sxsy(); ck = (round(sx,4), round(sy,4))
        if ck not in cache:
            cache[ck] = cv2.cvtColor(cv2.resize(base, (max(1,int(W*sx)), max(1,int(H*sy))),
                                     interpolation=cv2.INTER_AREA), cv2.COLOR_GRAY2BGR)
        return cache[ck]
    def page():
        big = canvas()
        return big.shape[0] if st["mode"] == "fit" else min(VIEW_H, big.shape[0])
    def d2o(x, y):
        sx, sy = sxsy(); return x/sx, (y + st["top"])/sy
    def o2d(x, y):
        sx, sy = sxsy(); return x*sx, y*sy - st["top"]

    def render():
        big = canvas(); ph = page()
        st["top"] = int(np.clip(st["top"], 0, max(0, big.shape[0]-ph)))
        view = big[st["top"]:st["top"]+ph].copy()
        sx, sy = sxsy(); cur = labels[st["i"]]
        for lab, s in shapes.items():
            col = (0,0,255) if lab == cur else ((0,220,255) if s["kind"] == "rect" else (0,230,0))
            if s["kind"] == "poly":
                pts = np.array([o2d(*p) for p in s["pts"]], np.int32)
                cv2.polylines(view, [pts], True, col, 2)
                tx, ty = pts[:,0].min(), pts[:,1].min()
            else:
                cx, cy = o2d(s["cx"], s["cy"])
                if s["kind"] == "ellipse":
                    axes, ang = ellipse_on_screen(s, sx, sy)
                    cv2.ellipse(view, (int(cx),int(cy)),
                                (max(1,int(axes[0])), max(1,int(axes[1]))), ang, 0, 360, col, 2)
                    cv2.drawMarker(view, (int(cx),int(cy)), col, cv2.MARKER_CROSS, 9, 1)
                    tx, ty = cx-axes[0], cy-axes[1]
                else:
                    ax, ay = s["ax"]*sx, s["ay"]*sy
                    cv2.rectangle(view, (int(cx-ax),int(cy-ay)), (int(cx+ax),int(cy+ay)), col, 2)
                    tx, ty = cx-ax, cy-ay
            cv2.putText(view, lab, (int(tx), int(ty)-6), cv2.FONT_HERSHEY_SIMPLEX,
                        .45, col, 1, cv2.LINE_AA)
        if st["poly"] and len(st["poly"]) > 1:
            cv2.polylines(view, [np.array([o2d(*p) for p in st["poly"]], np.int32)],
                          False, (255,140,0), 2)
        kind = st["kind"] or default_kind[cur]
        hud = (f"[{st['i']+1}/{len(labels)}] {cur}  mode={kind}  view={st['mode']}  "
               f"drawn {len(shapes)}/{len(labels)}")
        cv2.rectangle(view, (0,0), (view.shape[1], 24), (0,0,0), -1)
        cv2.putText(view, hud, (8,17), cv2.FONT_HERSHEY_SIMPLEX, .48, (255,255,255), 1, cv2.LINE_AA)
        cv2.imshow(key, view)

    def on_mouse(ev, x, y, flags, _):
        cur = labels[st["i"]]; kind = st["kind"] or default_kind[cur]
        ox, oy = d2o(x, y)
        if ev == cv2.EVENT_LBUTTONDOWN:
            if kind == "poly": st["poly"] = [(ox, oy)]
            else: st["draw"] = (ox, oy)                    # press point IS the centre
        elif ev == cv2.EVENT_MOUSEMOVE:
            if st["poly"] is not None and flags & cv2.EVENT_FLAG_LBUTTON:
                st["poly"].append((ox, oy)); render()
            elif st["draw"]:
                cx, cy = st["draw"]; ax, ay = abs(ox-cx), abs(oy-cy)
                if flags & cv2.EVENT_FLAG_SHIFTKEY:
                    sx, sy = sxsy(); rr = max(ax*sx, ay*sy); ax, ay = rr/sx, rr/sy
                shapes[cur] = dict(kind=kind, cx=float(cx), cy=float(cy),
                                   ax=float(max(ax,1)), ay=float(max(ay,1)), angle=0.0)
                render()
            elif st["move"] and flags & cv2.EVENT_FLAG_RBUTTON:
                s = shapes.get(cur)
                if s:
                    dx, dy = ox-st["move"][0], oy-st["move"][1]
                    if s["kind"] == "poly": s["pts"] = [(a+dx, b+dy) for a, b in s["pts"]]
                    else: s["cx"] += dx; s["cy"] += dy
                    st["move"] = (ox, oy); render()
        elif ev == cv2.EVENT_LBUTTONUP:
            if st["poly"] is not None:
                if len(st["poly"]) > 8:
                    shapes[cur] = dict(kind="poly", pts=[(float(a), float(b)) for a, b in st["poly"]])
                st["poly"] = None
            st["draw"] = None; render()
        elif ev == cv2.EVENT_RBUTTONDOWN: st["move"] = (ox, oy)
        elif ev == cv2.EVENT_RBUTTONUP:   st["move"] = None

    cv2.namedWindow(key, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(key, VIEW_W, min(VIEW_H, int(H*FIT[1])) + 8)
    cv2.setMouseCallback(key, on_mouse)
    render()
    while True:
        k = cv2.waitKey(20) & 0xFF
        if k == 255: continue
        cur = labels[st["i"]]; s = shapes.get(cur)
        if   k == 13: st["save"] = True; break
        elif k == 27: break                                   # ESC: skip this scan
        elif k == ord("q"): st["abort"] = True; break         # q: stop the whole run
        elif k == ord("n"): st["i"] = (st["i"]+1) % len(labels); st["kind"] = None
        elif k == ord("p"): st["i"] = (st["i"]-1) % len(labels); st["kind"] = None
        elif k == ord("e"): st["kind"] = "ellipse"
        elif k == ord("r"): st["kind"] = "rect"
        elif k == ord("g"): st["kind"] = "poly"
        elif k == ord("1"): st["mode"] = "fit"; st["top"] = 0
        elif k == ord("2"):
            st["mode"] = "one"
            if s and s["kind"] != "poly": st["top"] = int(max(0, s["cy"]*ONE[1] - VIEW_H/2))
        elif k == ord("j"): st["top"] += page()//2
        elif k == ord("k"): st["top"] -= page()//2
        elif k == ord("x"): shapes.pop(cur, None)
        elif s and s["kind"] != "poly":
            if   k == ord("w"): s["cy"] -= 1
            elif k == ord("s"): s["cy"] += 1
            elif k == ord("a"): s["cx"] -= 1
            elif k == ord("d"): s["cx"] += 1
            elif k == ord("W"): s["ay"] += 1
            elif k == ord("S"): s["ay"] = max(2, s["ay"]-1)
            elif k == ord("D"): s["ax"] += 1
            elif k == ord("A"): s["ax"] = max(2, s["ax"]-1)
            elif k == ord("["): s["angle"] = (s.get("angle",0)-2) % 360
            elif k == ord("]"): s["angle"] = (s.get("angle",0)+2) % 360
        render()
    cv2.destroyWindow(key)
    for _ in range(5): cv2.waitKey(1)
    if st["abort"]: return "ABORT"
    if not st["save"]: print(f"{key}: skipped, nothing saved"); return None
    miss = [l for l in labels if l not in shapes]
    if miss: print("nothing drawn for:", miss)
    return shapes

### Drawing all six scans

The loop below walks every scan that is not yet complete and opens the window for each in turn, saving
after each one so an interruption never costs you more than the scan you were on.

**Seeding.** A new scan starts pre-loaded with the shapes from an already-drawn scan of the **same day**,
because the dishes do not move between stir repeats — only the powder is disturbed. So for stir 2 and
stir 3 you are usually nudging rather than drawing. It only ever seeds within a day: day 1 is 2000 lines
and day 2 is 1700, so the geometry does not transfer across.

The seeded shapes still need checking. Re-stirring can shift a dish slightly and a plate that drifted a
few pixels will quietly pull table or rim into the mask.

**Per scan:** `ENTER` saves and moves to the next, `ESC` skips this one and moves on, `q` stops the run
entirely. Everything drawn so far is already on disk either way.

In [ ]:
ALL_ROIS = json.loads(ROI_FILE.read_text()) if ROI_FILE.exists() else {}
SEED_FROM_PREVIOUS = True
REDRAW = []          # force a redraw of these scans, e.g. ["stir2_SWIR_384_..."]

def roi_status():
    rows = []
    for key in CUBES:
        need = set(["WHITE"] + INFO[key]["plates"])
        have = set(ALL_ROIS.get(key, {}))
        rows.append(dict(scan=key, day=INFO[key]["day"], stir=INFO[key]["capture"],
                         drawn=len(have & need), needed=len(need),
                         missing=", ".join(sorted(need - have)) or "-",
                         complete=need.issubset(have)))
    return pd.DataFrame(rows)


def seed_for(key):
    # shapes from another scan of the same day, as a starting point
    if not SEED_FROM_PREVIOUS: return None
    for other, sh in ALL_ROIS.items():
        if other == key or other not in INFO: continue
        if INFO[other]["day"] == INFO[key]["day"] and INFO[other]["plates"] == INFO[key]["plates"]:
            if set(["WHITE"] + INFO[key]["plates"]).issubset(sh):
                return {k: dict(v) for k, v in sh.items()}, other
    return None


def draw_all():
    st = roi_status()
    todo = [r.scan for _, r in st.iterrows() if (not r.complete) or (r.scan in REDRAW)]
    if not todo:
        print("every scan is complete -- nothing to draw"); return
    print(f"{len(todo)} scan(s) to draw:", *todo, sep="\n   ")
    for n, key in enumerate(todo, 1):
        existing = ALL_ROIS.get(key)
        note = ""
        if not existing:
            sd = seed_for(key)
            if sd:
                existing, src = sd
                note = f"  (seeded from {src} -- check every shape)"
        print(f"\n[{n}/{len(todo)}] {key}{note}")
        res = draw_rois(CUBES[key], key, INFO[key]["plates"], existing=existing)
        if res == "ABORT":
            print("run stopped. Progress so far is saved."); break
        if res is None:
            continue
        ALL_ROIS[key] = res
        ROI_FILE.write_text(json.dumps(ALL_ROIS, indent=1))
        print(f"   saved {len(res)} shapes")

draw_all()
print()
print(roi_status().to_string(index=False))

In [ ]:
# Check before going further -- an incomplete scan silently drops plates from the CSV.
status = roi_status()
bad = status[~status.complete]
if len(bad):
    print("INCOMPLETE -- rerun draw_all() for these:")
    print(bad.to_string(index=False))
else:
    n_plates = sum(len(INFO[k]["plates"]) for k in ALL_ROIS)
    print(f"all {len(ALL_ROIS)} scans complete: {n_plates} plates total")
    print(status.groupby(["day"]).agg(scans=("scan", "count"),
                                      plates=("needed", lambda v: int(sum(v)-len(v)))).to_string())

### Fallback if the window will not open

Headless OpenCV build, remote session, no display. Same storage format, so everything downstream is
identical, and section 5 renders whatever you put here.

In [ ]:
# ALL_ROIS["scan_stem"] = {
#     "WHITE":  dict(kind="rect",    cx=192, cy=150, ax=180, ay=35, angle=0),
#     "AB100":  dict(kind="ellipse", cx=220, cy=430, ax=88,  ay=58, angle=0),
#     "CH100":  dict(kind="poly",    pts=[[150,600],[290,610],[300,700],[160,695]]),
# }
# ROI_FILE.write_text(json.dumps(ALL_ROIS, indent=1))
print("current:", {k: sorted(v) for k, v in ALL_ROIS.items()})

## 4. White reference and plate masks

The white reference is built **per across-track column** from the rows you drew. Your strip spans the
swath, so the lamp and lens profile is measured at every column rather than assumed — which matters here
because the plates sit at different across-track positions, and a single averaged white would leave that
profile in the spectra as a position-dependent error.

Inside each plate oval, three further rejections:

- pixels below 55 % of the region median — rim or shadow that crept inside the shape;
- pixels above the 99.7th percentile, dilated by one pixel — specular highlights and their fringes;
- an erosion of `ERODE_PX` to pull the boundary away from the edge.

Freehand outlines skip the erosion, since a traced boundary is already deliberate.

In [ ]:
def shape_mask(shape, H, W):
    m = np.zeros((H, W), np.uint8)
    if shape["kind"] == "poly":
        cv2.fillPoly(m, [np.array([[int(round(a)), int(round(b))] for a, b in shape["pts"]],
                                  np.int32)], 1)
        return m.astype(bool)
    cx, cy = int(round(shape["cx"])), int(round(shape["cy"]))
    ax, ay = int(round(shape["ax"])), int(round(shape["ay"]))
    if shape["kind"] == "ellipse":
        cv2.ellipse(m, (cx, cy), (max(ax,1), max(ay,1)), shape.get("angle", 0), 0, 360, 1, -1)
    else:
        cv2.rectangle(m, (cx-ax, cy-ay), (cx+ax, cy+ay), 1, -1)
    return m.astype(bool)


def clean_mask(mask, bright, kind="ellipse", erode=ERODE_PX, dark_frac=0.55, spec_q=99.7):
    m = mask.astype(np.uint8)
    e = 0 if kind == "poly" else erode
    if e > 0:
        m = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*e+1, 2*e+1)))
    m = m.astype(bool)
    if m.sum() < 30: return m
    m &= bright > dark_frac * np.nanmedian(bright[m])
    if m.sum() > 50:
        hi = np.nanpercentile(bright[m], spec_q)
        m &= ~cv2.dilate((bright > hi).astype(np.uint8), np.ones((3,3), np.uint8)).astype(bool)
    return m


def build_white(cube, shape, bad_frac=0.5):
    r0, r1 = int(shape["cy"] - shape["ay"]), int(shape["cy"] + shape["ay"])
    c0, c1 = int(shape["cx"] - shape["ax"]), int(shape["cx"] + shape["ax"])
    r0, c0 = max(0, r0), max(0, c0)
    r1, c1 = min(cube.shape[0], r1), min(cube.shape[1], c1)
    S = cube.shape[1]
    W = np.full((S, cube.wl.size), np.nan)
    W[c0:c1] = np.nanmedian(cube.region(r0, r1, c0, c1), axis=0)
    ref = np.nanmedian(W[c0:c1], axis=0)
    lvl = np.nanmedian(W / np.maximum(ref, 1e-9), axis=1)
    bad = ~np.isfinite(lvl) | (lvl < bad_frac)
    good = np.where(~bad)[0]
    if bad.any() and good.size > 2:
        for jb in range(W.shape[1]):
            W[bad, jb] = np.interp(np.where(bad)[0], good, W[good, jb])
    return W, (r0, r1, c0, c1), int(bad.sum()), (c1 - c0) / S


WHITE, MASKS = {}, {}
for key in ALL_ROIS:
    cube = CUBES[key]; H, Wd = cube.shape[0], cube.shape[1]
    if "WHITE" not in ALL_ROIS[key]:
        print(f"{key}: no WHITE strip drawn -- skipped"); continue
    Wc, wbox, nbad, cov = build_white(cube, ALL_ROIS[key]["WHITE"])
    WHITE[key] = Wc
    print(f"{key}: white rows {wbox[0]}-{wbox[1]}, spans {cov*100:.0f}% of the swath, "
          f"{nbad} columns repaired")
    br = cube.brightness(step=1)
    mk = {}
    for lab, sh in ALL_ROIS[key].items():
        if lab == "WHITE": continue
        raw = shape_mask(sh, H, Wd)
        cl = clean_mask(raw, br, kind=sh["kind"])
        mk[lab] = dict(raw=raw, mask=cl, n=int(cl.sum()),
                       keep=float(cl.sum()/max(raw.sum(), 1)))
        print(f"    {lab:10s} {cl.sum():7d} px ({mk[lab]['keep']*100:.0f}% of the shape)")
    MASKS[key] = mk

In [ ]:
def review(key, cols=3):
    br = CUBES[key].brightness(step=1)
    items = list(MASKS[key].items())
    rows = int(np.ceil(len(items)/cols))
    fig, axs = plt.subplots(rows, cols, figsize=(3.4*cols, 2.7*rows), squeeze=False)
    for ax in axs.ravel(): ax.axis("off")
    for jx, (lab, d) in enumerate(items):
        r, c = divmod(jx, cols); ax = axs[r][c]
        ys, xs = np.where(d["raw"])
        if ys.size == 0: continue
        p = 12
        r0, r1 = max(0, ys.min()-p), min(br.shape[0], ys.max()+p)
        c0, c1 = max(0, xs.min()-p), min(br.shape[1], xs.max()+p)
        sub = np.nan_to_num(br[r0:r1, c0:c1])
        lo, hi = np.nanpercentile(sub, [2, 98])
        g = np.clip((sub-lo)/max(hi-lo, 1e-9), 0, 1)
        rgb = np.dstack([g, g, g]); mm = d["mask"][r0:r1, c0:c1]
        rgb[...,0] = np.where(mm, np.clip(g*.5+.5, 0, 1), g*.8)
        rgb[...,1] = np.where(mm, g*.42, g*.8)
        rgb[...,2] = np.where(mm, g*.32, g*.8)
        ax.imshow(np.clip(rgb, 0, 1)); ax.axis("off")
        ax.set_title(f"{lab}  {d['n']} px ({d['keep']*100:.0f}%)", fontsize=8)
    fig.suptitle(f"{key}: mask review (red = kept)", fontsize=9)
    plt.tight_layout(); plt.savefig(OUT / f"{key}_masks.png", dpi=125, bbox_inches="tight"); plt.show()

for key in MASKS: review(key)

## 5. Reflectance and patch sampling

    R(row, col, band) = L(row, col, band) / W(col, band) × R_certified_white(band)

applied inside each plate mask only — converting whole 880 MB cubes when most of the frame is table would
just burn memory.

Patches are laid on a grid with spacing `STRIDE`. With `STRIDE == PATCH` they do not overlap, which is the
right default: overlapping patches share pixels and so are even less independent than they already are.
A patch is kept only if `MIN_VALID` of its pixels fall inside the mask, so no patch straddles the edge.

In [ ]:
def plate_reflectance(key, lab):
    cube = CUBES[key]; d = MASKS[key][lab]
    ys, xs = np.where(d["mask"])
    if ys.size == 0: return None
    r0, r1, c0, c1 = ys.min(), ys.max()+1, xs.min(), xs.max()+1
    L = cube.region(r0, r1, c0, c1)
    W = WHITE[key][c0:c1, :][None, :, :]
    R = L / np.maximum(W, 1e-9) * R_WHITE[None, None, :]
    R[(R < -0.05) | (R > 1.6)] = np.nan
    return R, d["mask"][r0:r1, c0:c1], (int(r0), int(c0))


def sample_patches(R, m, patch=PATCH, stride=STRIDE, min_valid=MIN_VALID,
                   max_n=MAX_PER_PLATE, rng=RNG):
    H, Wd, B = R.shape
    specs, pos = [], []
    for i in range(0, H-patch+1, stride):
        for jx in range(0, Wd-patch+1, stride):
            sm = m[i:i+patch, jx:jx+patch]
            if sm.mean() < min_valid: continue
            s = np.nanmean(R[i:i+patch, jx:jx+patch, :][sm], axis=0)
            if np.all(np.isfinite(s)):
                specs.append(s); pos.append((i+patch//2, jx+patch//2, int(sm.sum())))
    if not specs: return np.empty((0, R.shape[2])), []
    if max_n and len(specs) > max_n:
        keep = np.sort(rng.choice(len(specs), max_n, replace=False))
        specs = [specs[k] for k in keep]; pos = [pos[k] for k in keep]
    return np.array(specs), pos


X, rows = [], []
for key in MASKS:
    for lab in INFO[key]["plates"]:
        if lab not in MASKS[key]: continue
        res = plate_reflectance(key, lab)
        if res is None: continue
        R, m, (r0, c0) = res
        specs, pos = sample_patches(R, m)
        if specs.shape[0] == 0:
            print(f"{key} {lab}: no valid patches -- lower PATCH or MIN_VALID"); continue
        X.append(specs)
        info = parse_label(lab)
        for si, (pr, pc, npx) in enumerate(pos):
            rows.append(dict(sample_id=f"{key}|{lab}|{si:04d}", scan=key, day=INFO[key]["day"],
                             capture=INFO[key]["capture"], label=lab, **info,
                             group=f"{key}|{lab}", plate=f"day{INFO[key]['day']}|{lab}",
                             patch_row=r0+pr, patch_col=c0+pc, n_pixels=npx))
        print(f"{key[:28]:28s} {lab:10s} {int(m.sum()):6d} px -> {specs.shape[0]:4d} patches")

X = np.vstack(X); META = pd.DataFrame(rows)
print(f"\n{X.shape[0]} samples x {X.shape[1]} bands from {META['scan'].nunique()} scans")
print(META.groupby(["day", "capture"]).size().rename("samples").to_string())
print(f"patch = {PATCH}x{PATCH} px ~ {PATCH*MM_PER_PX:.1f} x {PATCH*MM_PER_PX:.1f} mm")
if PATCH * MM_PER_PX < 5:
    print("!! Patches under ~5 mm are dominated by particle lottery rather than composition.")
    print("   Raise PATCH, or accept that individual samples will be noisy.")

## 6. Splits

Four columns are added, and they are not interchangeable. Which one you use determines whether your
reported accuracy means anything.

**`role`** — what each plate is for.

| role | plates | why |
|---|---|---|
| `calibration` | RB100 and RB[90:10]…RB[50:50] | chicory 0–50 %, the model's working range |
| `extrapolation_check` | CH100 | chicory 100 %. Excluded from fitting: there is a 50-point gap below it, and one plate at that leverage would dictate the slope of the whole model. Predict it, don't train on it. |
| `unknown` | A, B, C | what you actually want to know |
| `out_of_series` | AB100 | arabica, not robusta. A good negative control — a robusta model should flag it as out-of-domain. |

**`fold_level`** — leave-one-concentration-out. Every distinct `chicory_pct` in the calibration set is its
own fold. **This is the split to report.** It is the only one where the held-out concentration is genuinely
absent from training, so it answers "does the model track composition?" rather than "can it recognise a
dish it has seen?"

**`fold_plate`** — grouped k-fold by plate. More conventional, and always more flattering than
`fold_level`, because another plate at the same concentration is still in training. Useful as a secondary
number; misleading if reported alone.

**`fold_capture`** — leave-one-stir-out, now that you have three repeats of every scan. Train on stirs 1
and 2, test on stir 3. This measures **repeatability**: how much a prediction moves when the same powder
is re-stirred and re-sampled. It is a useful number and it is not validation — the same concentrations are
in training, so it says nothing about whether the model tracks composition. Reporting it as validation
would overstate what you have.

**`split_holdout`** — a ready-made single train/test split holding out one interior concentration, for
when you just want to try something quickly.

There is no random-split column, deliberately. With patches this correlated a random split would report
something close to perfect and mean nothing at all.

**One more thing the splits cannot fix.** Your calibration mixtures (10–50 %) are all from day 2, while
RB100 at 0 % and the unknowns are from day 1. No plate appears on both days, so any session difference —
lamp warm-up, focus, height — is mathematically indistinguishable from a concentration effect. The `day`
column is in the CSV so you can test for it: fit on day 2 alone, predict day-1 RB100, and see whether it
comes back near 0 %.

In [ ]:
CAL_MAX_CHICORY = 50.0        # calibration range: 0-50% chicory
HOLDOUT_LEVEL   = 30.0        # interior level reserved by split_holdout
N_PLATE_FOLDS   = 5

def assign_role(r):
    if r["unknown"]: return "unknown"
    if r["species"] == "arabica": return "out_of_series"
    if r["species"] == "chicory": return "extrapolation_check"
    return "calibration" if r["chicory_pct"] <= CAL_MAX_CHICORY else "extrapolation_check"

META["role"] = META.apply(assign_role, axis=1)

cal = META["role"] == "calibration"
levels = sorted(META.loc[cal, "chicory_pct"].unique())
lmap = {v: i for i, v in enumerate(levels)}
# only calibration rows get a fold -- AB100 also sits at 0% chicory and would otherwise
# be swept into fold 0 despite being arabica
META["fold_level"] = pd.NA
META.loc[cal, "fold_level"] = META.loc[cal, "chicory_pct"].map(lmap)
META["fold_level"] = META["fold_level"].astype("Int64")

groups = sorted(META.loc[cal, "group"].unique())
gmap = {g: i % N_PLATE_FOLDS for i, g in enumerate(groups)}
META["fold_plate"] = pd.NA
META.loc[cal, "fold_plate"] = META.loc[cal, "group"].map(gmap)
META["fold_plate"] = META["fold_plate"].astype("Int64")

META["fold_capture"] = pd.NA
META.loc[cal, "fold_capture"] = META.loc[cal, "capture"]
META["fold_capture"] = META["fold_capture"].astype("Int64")

META["split_holdout"] = np.where(~cal, "excluded",
                        np.where(META["chicory_pct"] == HOLDOUT_LEVEL, "test", "train"))

print("samples by role:")
print(META.groupby("role").size().to_string())
print("\ncalibration levels (% chicory) and their folds:")
print(META[cal].groupby(["chicory_pct", "fold_level", "day"]).size()
      .rename("n_samples").reset_index().to_string(index=False))
print("\nsamples per stir repeat (calibration only):")
print(META[cal].groupby(["capture"]).size().rename("n").to_string())
print("\nplates per concentration (should be one per stir repeat):")
print(META[cal].groupby("chicory_pct")["group"].nunique().rename("n_plates").to_string())
print("\nsplit_holdout:")
print(META.groupby("split_holdout").size().to_string())
print(f"\nCalibration spans {min(levels):.0f}-{max(levels):.0f}% chicory across {len(levels)} levels "
      f"and {len(groups)} plates.")
print("Predictions outside that range are extrapolation, whatever number the model returns.")

## 7. The CSV

In [ ]:
wl_cols = [str(w) for w in WL_R]
META_COLS = ["sample_id", "scan", "day", "capture", "label", "plate", "species",
             "robusta_pct", "chicory_pct", "unknown", "role", "group",
             "fold_level", "fold_plate", "fold_capture", "split_holdout",
             "patch_row", "patch_col", "n_pixels"]

df = pd.concat([META[META_COLS].reset_index(drop=True),
                pd.DataFrame(np.round(X, 6), columns=wl_cols)], axis=1)
df.to_csv(OUT / "training_samples.csv", index=False)

df[df.role == "calibration"].to_csv(OUT / "training_calibration.csv", index=False)
df[df.role == "unknown"].to_csv(OUT / "training_unknowns.csv", index=False)

pd.DataFrame({"band": np.arange(1, WL.size+1), "wavelength_exact_nm": np.round(WL, 4),
              "wavelength_rounded_nm": WL_R, "white_certified": np.round(R_WHITE, 5),
              "beyond_certificate": OUTSIDE}).to_csv(OUT / "wavelengths.csv", index=False)

np.savez_compressed(OUT / "training_samples.npz", X=X, wl=WL, wl_rounded=WL_R,
                    meta=META[META_COLS].to_records(index=False))

print(f"training_samples.csv: {df.shape[0]} rows x {df.shape[1]} columns "
      f"({len(META_COLS)} metadata + {len(wl_cols)} bands)")
for f in sorted(OUT.glob("*.csv")): print("   ", f.name)
print()
print(df[META_COLS[:14]].head(6).to_string(index=False))
print("\nfirst wavelength columns:", wl_cols[:6], "...", wl_cols[-3:])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
cmap = plt.get_cmap("viridis")
for lab, g in META[META["role"] == "calibration"].groupby("label"):
    mu = X[g.index].mean(0)
    axs[0].plot(WL, mu, lw=1.2, color=cmap(g["chicory_pct"].iloc[0]/60),
                label=f"{lab} ({g['chicory_pct'].iloc[0]:.0f}%)")
for lab, g in META[META["role"] != "calibration"].groupby("label"):
    axs[0].plot(WL, X[g.index].mean(0), lw=1.0, ls="--", label=lab)
axs[0].set_title("mean of the patch samples per plate"); axs[0].legend(fontsize=6, ncol=2)

big = META[META["role"] == "calibration"].groupby("group").size().idxmax()
sel = (META["group"] == big).values
for s in X[sel][:80]:
    axs[1].plot(WL, s, lw=.4, alpha=.35, color="tab:blue")
axs[1].plot(WL, X[sel].mean(0), lw=1.8, color="crimson")
axs[1].set_title(f"spread within one plate: {big.split('|')[-1]} ({sel.sum()} samples)")

for ax in axs:
    for nm, t in [(1450,"O-H"), (1730,"C-H lipid"), (1940,"O-H"),
                  (2100,"C-O carb."), (2310,"C-H comb.")]:
        ax.axvline(nm, color="0.75", lw=.8, ls=":")
        ax.text(nm, ax.get_ylim()[1], t, fontsize=6, rotation=90, va="top")
    ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("reflectance"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "samples_overview.png", dpi=130, bbox_inches="tight"); plt.show()

sd_within = float(np.mean([X[(META["group"] == g).values].std(0).mean()
                           for g in META["group"].unique()]))
sd_between = float(np.mean(np.std([X[(META["label"] == l).values].mean(0)
                                   for l in META[META.role=="calibration"].label.unique()], axis=0)))
print(f"mean within-plate spread:  {sd_within:.4f} reflectance units")
print(f"spread between calibration levels: {sd_between:.4f}")
print("\nIf the within-plate spread is comparable to the between-level spread, the composition")
print("signal is buried in surface texture -- raise PATCH so each sample averages more particles.")

## 8. Using it

```python
import pandas as pd, numpy as np
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.cross_decomposition import PLSRegression

df = pd.read_csv("samples_out/training_samples.csv")
wl = [c for c in df.columns if c.replace(".", "", 1).isdigit()]

cal = df[df.role == "calibration"]
X, y = cal[wl].values, cal["chicory_pct"].values

# leave-one-concentration-out -- the split to report
logo = LeaveOneGroupOut()
pred = np.full(len(y), np.nan)
for tr, te in logo.split(X, y, groups=cal["fold_level"]):
    pred[te] = PLSRegression(n_components=3).fit(X[tr], y[tr]).predict(X[te]).ravel()

print("RMSECV:", np.sqrt(np.mean((pred - y) ** 2)))

# repeatability: same thing over fold_capture instead
rep = np.full(len(y), np.nan)
for tr, te in logo.split(X, y, groups=cal["fold_capture"]):
    rep[te] = PLSRegression(n_components=3).fit(X[tr], y[tr]).predict(X[te]).ravel()
print("leave-one-stir-out RMSE:", np.sqrt(np.mean((rep - y) ** 2)), "(repeatability, not validation)")
```

Before trusting any number from that, two checks worth ten minutes each:

**Permutation test.** With five or six distinct plates in calibration, a decent RMSECV can arise by
chance — the model may just be separating a handful of objects and happening to order them. Shuffle
`chicory_pct` across whole plates, refit, and see how often the null beats your real score.

**The day test.** Fit on day 2 only, predict day-1 RB100. It should come back near 0 % chicory. If it comes
back at 15 %, that is your session offset measured in the units you care about, and every A/B/C prediction
carries it.

**Repeatability, now that you have three stirs.** Predict each plate from each stir separately and look at
the spread across the three. That spread is the floor on your precision — no model can be more certain
about a sample than re-stirring it makes the measurement. Quote it alongside RMSECV.

And when you do predict A, B and C: check they fall inside 0–50 % chicory first. Outside that range the
model is extrapolating across a gap it has no data for, and it will still return a confident number.